Import libraries


In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd().parent))

from src.data.load import load_datasets_from_yaml
from src.data.dtypes import cast_datasets

Load datasets

In [ ]:
datasets = load_datasets_from_yaml("../config/datasets.yaml")

Converting variables dtypes

In [ ]:
datasets = cast_datasets(datasets=datasets)

Datasets sizes and variables dtypes after conversion

In [ ]:
for name, df in datasets.items():
    print(f"\n{name}: {df.shape}")
    print(df.dtypes)

Common heats between datasets

In [ ]:
heatid_sets = {
    name: set(df["HEATID"].unique())
    for name, df in datasets.items()
    if "HEATID" in df.columns
}

print(f"{'Dataset':<25} {'unique HEATIDs':>15}")
print("-" * 42)
for name, ids in heatid_sets.items():
    print(f"{name:<25} {len(ids):>15}")

# Interseção de todos — quantos heats têm dados em TODOS os datasets
all_common = set.intersection(*heatid_sets.values())
print(f"\n{'Heats with complete data:':<25} {len(all_common):>15}")

# Pair-wise: quais datasets têm cobertura diferente do principal (eaf_temp)
if "eaf_temp" in heatid_sets:
    ref = heatid_sets["eaf_temp"]
    print(f"\nHEATIDs in eaf_temp but missing in:")
    for name, ids in heatid_sets.items():
        if name != "eaf_temp":
            diff = ref - ids
            if diff:
                print(f"  {name:<23} → {len(diff)} heats without data")

Datasets missing values

In [ ]:
for name, df in datasets.items():
    missing = df.isnull().mean().mul(100).round(1)
    missing = missing[missing > 0]
    if not missing.empty:
        print(f"\n{name}")
        print(missing)

Number of heats with chemical composition measurements but missing nitrogen measurement

In [ ]:
df = datasets["eaf_final_chemical"]  # ajustar nome se necessário
heat_missing = df.groupby("HEATID")["VALN"].apply(lambda x: x.isna().all())
heat_missing.mean()  # proporção de heats sem nenhuma medição de N
heat_missing.value_counts()

Rows per Heat (HEATID): min, median, and max by dataset

In [ ]:
print(f"{'Dataset':<25} {'Min linhas/heat':>16} {'Mediana':>9} {'Max':>6}")
print("-" * 60)
for name, df in datasets.items():
    if "HEATID" not in df.columns:
        continue
    counts = df.groupby("HEATID").size()
    print(f"{name:<25} {counts.min():>16} {counts.median():>9.1f} {counts.max():>6}")

Temporal coverage of datasets

In [ ]:
for name, df in datasets.items():
    dt_cols = df.select_dtypes(include="datetime64").columns.tolist()
    for col in dt_cols:
        mn, mx = df[col].min(), df[col].max()
        print(f"{name}.{col}: {mn.date()} to {mx.date()} ({(mx - mn).days} days)")

Datasets basic statistics


In [ ]:
for name, df in datasets.items():
    print(f"\nDataset: {name}")
    display(df.describe())

### Relational structure of the datasets

All datasets are related through the `HEATID` key.

| Dataset              | Granularity          | Join key | Type            |
|----------------------|----------------------|----------|-----------------|
| `eaf_temp`           | measurement per heat | HEATID   | transactional   |
| `eaf_final_chemical` | 1 row per heat       | HEATID   | static          |
| `eaf_transformer`    | N rows per heat      | HEATID   | time series     |
| `eaf_gaslance`       | N rows per heat      | HEATID   | time series     |
| `inj_mat`            | N rows per heat      | HEATID   | time series     |
| `ladle_tapping`      | N rows per heat      | HEATID   | transactional   |
| `basket_charged`     | N rows per heat      | HEATID   | transactional   |
| `eaf_added`          | N rows per heat      | HEATID   | transactional   |
| `ferro`              | material reference   | MAT_CODE | lookup table    |

### Saving pre-processed data

In [ ]:
for name, df in datasets.items():
    df.to_csv(f"../data/interim/{name}.csv", index=False)